# Collections Persona Segmentation — K-Means vs Shallow Decision Tree

**The problem.** We have ~1 million customers who are behind on payments. A collections team cannot
treat a million people individually, so we want to sort them into a handful of *personas*
(for example "recently slipped but usually pays", "deep in arrears, unresponsive") and design a
collection strategy per persona.

**Two ways to build the personas.**

| | K-Means | Shallow decision tree |
|---|---|---|
| Looks at | Only the customer's behaviour features | Behaviour features **and** whether the customer later paid |
| Groups customers who… | *look alike* | *behave alike in outcome* (pay vs don't pay) |
| Output | Cluster number + centroid | A plain rule such as `days_past_due > 70` |
| Needs an outcome label? | No | Yes |

**How we judge them.** Both are built on 70% of customers and tested on the untouched 30%.
We compare them on (a) textbook clustering metrics and (b) the thing the business cares about:
does the persona actually predict who pays? To make (b) honest we also score a **random segmentation**
as a floor — any real method must clearly beat random.

**Notebook flow:** Constants → Prepare data → Split → K-Means (elbow) → Score validation →
Evaluate → Decision tree → Compare (vs random) → Recommendation → Standalone scoring pipeline.

## 1. Constants

Everything you might want to change lives here, in CAPITALS, so nobody has to hunt through the code.

In [ ]:
import json, os, pickle
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, roc_auc_score

# ---- Paths -------------------------------------------------------------------
PATH_INPUT         = "customers.csv"            # one row per customer, columns listed below
PATH_ARTIFACTS     = "artifacts"                # folder where we save everything the scoring job needs
PATH_SCALER        = f"{PATH_ARTIFACTS}/scaler.json"      # medians, clip bounds, mean, std per feature
PATH_CENTROIDS     = f"{PATH_ARTIFACTS}/centroids.json"   # K-Means cluster centres + names
PATH_TREE          = f"{PATH_ARTIFACTS}/tree.pkl"          # the fitted decision tree
PATH_FINAL_CHOICE  = f"{PATH_ARTIFACTS}/final_choice.json" # which method we recommended

# ---- Columns ---------------------------------------------------------------------
ID_COL = "customer_id"

INPUT_FEATURES = [            # what the customer LOOKS like today -> used to BUILD the segments
    "days_past_due",          # how many days late the oldest unpaid amount is
    "outstanding_balance",    # how much they owe
    "utilization_pct",        # balance / credit limit
    "missed_payments_12m",    # how many payments missed in the last year
    "tenure_months",          # how long they have been a customer
    "monthly_income",
    "contact_attempts_30d",   # how many times collections tried to reach them last month
    "promise_kept_rate",      # of the times they promised to pay, how often they did
    "last_payment_ratio",     # last payment amount / amount that was due
]
PERFORMANCE_VARS = [          # what HAPPENED afterwards -> used only to JUDGE the segments, never to build K-Means
    "recovered_90d",          # 1 if the customer paid up within 90 days, else 0
    "recovery_rate_90d",      # share of the balance recovered within 90 days (0 to 1)
]
TARGET = "recovered_90d"      # the single outcome the tree learns from and all lift numbers use

# ---- Data preparation -----------------------------------------------------------
CLIP_LOWER_PCT = 0.01         # values below the 1st percentile are pulled up to it
CLIP_UPPER_PCT = 0.99         # values above the 99th percentile are pulled down to it
TRAIN_FRACTION = 0.70
RANDOM_SEED    = 42

# ---- K-Means --------------------------------------------------------------------
K_CANDIDATES    = range(2, 11)  # try 2..10 clusters and pick the best
K_MINIMUM       = 3             # fewer than 3 personas is too crude for a collections strategy
KMEANS_N_INIT   = 3             # K-Means starts from random points; run 3 times, keep the best
ELBOW_ROWS      = 200_000       # the elbow sweep runs 9 fits, so use a subsample for speed
METRIC_ROWS     = 20_000        # silhouette compares every pair of rows (slow), so subsample

# ---- Decision tree ----------------------------------------------------------------
TREE_MAX_DEPTH     = 3          # 3 yes/no questions deep -> at most 8 leaves (segments)
TREE_MIN_LEAF_FRAC = 0.05       # no segment smaller than 5% of customers (too small to build a strategy around)

np.random.seed(RANDOM_SEED)
os.makedirs(PATH_ARTIFACTS, exist_ok=True)
print(f"{len(INPUT_FEATURES)} input features, {len(PERFORMANCE_VARS)} performance variables, target = {TARGET}")

## 2. Prepare data

Three things can break a clustering algorithm, and we fix each one:

1. **Missing values** — K-Means cannot compute a distance to a blank cell. We fill blanks with the
   feature's *median* (middle value), which is not thrown off by extreme values the way the mean is.
2. **Outliers** — one customer with a 100× balance pulls a cluster centre towards them alone.
   We *winsorize*: anything beyond the 1st/99th percentile is clipped to that percentile.
3. **Scale** — K-Means measures straight-line distance. If balance ranges 0–50,000 and promise-kept
   rate ranges 0–1, the balance dominates every distance and the rate is ignored. We *standardize*
   every feature to mean 0, standard deviation 1, so all features count equally.

We **save** the medians, clip bounds, means and stds to disk, because new customers must be
transformed with exactly these numbers — not re-computed on the new data — or the centroids no longer apply.

In [ ]:
customers = pd.read_csv(PATH_INPUT)
print(f"Loaded {len(customers):,} customers x {customers.shape[1]} columns")
missing_cols = set(INPUT_FEATURES + PERFORMANCE_VARS + [ID_COL]) - set(customers.columns)
assert not missing_cols, f"Input file is missing columns: {missing_cols}"

print("\nRaw feature summary (look for odd mins/maxes and large ranges):")
customers[INPUT_FEATURES].describe().T.round(2)

In [ ]:
# ---- Step 1: missing values -> median ---------------------------------------------
missing_pct = customers[INPUT_FEATURES].isna().mean().mul(100).round(2)
print("Missing % per feature (only features with any gaps shown):")
print(missing_pct[missing_pct > 0].to_string() if (missing_pct > 0).any() else "  none")

feature_medians = customers[INPUT_FEATURES].median()          # one number per feature
customers[INPUT_FEATURES] = customers[INPUT_FEATURES].fillna(feature_medians)

# ---- Step 2: outliers -> clip to 1st / 99th percentile --------------------------------
lower_bounds = customers[INPUT_FEATURES].quantile(CLIP_LOWER_PCT)
upper_bounds = customers[INPUT_FEATURES].quantile(CLIP_UPPER_PCT)
is_outlier = (customers[INPUT_FEATURES] < lower_bounds) | (customers[INPUT_FEATURES] > upper_bounds)
print("\nRows clipped per feature (expect roughly 2% each):")
print(is_outlier.sum().to_string())
customers[INPUT_FEATURES] = customers[INPUT_FEATURES].clip(lower_bounds, upper_bounds, axis=1)

# ---- Step 3: scale -> z-score = (value - mean) / std ----------------------------------
feature_ranges = customers[INPUT_FEATURES].max() - customers[INPUT_FEATURES].min()
print(f"\nWidest feature range is {feature_ranges.max()/feature_ranges.min():,.0f}x the narrowest "
      f"({feature_ranges.idxmax()} vs {feature_ranges.idxmin()}) -> scaling is required.")

scaler_mean = customers[INPUT_FEATURES].mean()
scaler_std  = customers[INPUT_FEATURES].std()
scaled_features = (customers[INPUT_FEATURES] - scaler_mean) / scaler_std

# Quick check that scaling worked: every column should now be centred at 0 with spread 1
print(f"After scaling: largest |mean| = {scaled_features.mean().abs().max():.4f}, stds = {scaled_features.std().round(2).unique()}")

# ---- Save every number the scoring job will need to repeat steps 1-3 identically ------------
scaler_attributes = {
    "features": INPUT_FEATURES,
    "median":   feature_medians.to_dict(),
    "lower":    lower_bounds.to_dict(),
    "upper":    upper_bounds.to_dict(),
    "mean":     scaler_mean.to_dict(),
    "std":      scaler_std.to_dict(),
}
json.dump(scaler_attributes, open(PATH_SCALER, "w"), indent=2)
print(f"\nSaved scaling attributes -> {PATH_SCALER}")

## 3. Split 70 : 30

We build the segments on 70% of customers (**train**) and keep 30% aside (**validation**) that the
models never see. If the segments behave the same on the validation set as on train, they are
genuine patterns and not an accident of the particular customers we fitted on.

`X_*` holds the scaled numbers the algorithms consume; `raw_*` holds the original columns
(IDs, outcomes, unscaled values) so we can read results in plain units.

In [ ]:
is_train = np.random.rand(len(customers)) < TRAIN_FRACTION    # each row gets a 70% chance of landing in train

X_train = scaled_features[is_train].to_numpy()
X_valid = scaled_features[~is_train].to_numpy()
raw_train = customers[is_train].reset_index(drop=True)
raw_valid = customers[~is_train].reset_index(drop=True)

print(f"Train: {len(X_train):,} customers   Validation: {len(X_valid):,} customers")
print(f"Overall recovery rate   train = {raw_train[TARGET].mean():.3f}   valid = {raw_valid[TARGET].mean():.3f}   (should be close)")

## 4 & 5. K-Means — how many clusters?

K-Means needs to be told the number of clusters `k` up front. Too few and different personas get
merged; too many and we split hairs. We fit K-Means for every `k` from 2 to 10 and look at two numbers:

* **Inertia** — total squared distance of every customer to its cluster centre. Lower is tighter.
  It *always* falls as `k` rises, so we look for the "elbow": the point after which adding a
  cluster buys very little extra tightness.
* **Silhouette** — for each customer, how much closer am I to my own cluster than to the nearest
  other cluster? Ranges −1 to 1; higher means clusters are well separated. Unlike inertia it can go
  down with more clusters, so it tells us when we have gone too far.

We choose the `k` with the best silhouette, with a floor of `K_MINIMUM` (silhouette almost always
loves `k = 2`, which is too coarse to act on).

In [ ]:
# Subsample train for the sweep: 9 fits on 700k rows would be slow and the elbow shape doesn't change
elbow_rows   = X_train[np.random.choice(len(X_train), ELBOW_ROWS, replace=False)]
metric_rows  = elbow_rows[:METRIC_ROWS]       # silhouette is O(n^2), so use an even smaller slice

elbow_results = []
for k in K_CANDIDATES:
    model = KMeans(n_clusters=k, n_init=KMEANS_N_INIT, random_state=RANDOM_SEED).fit(elbow_rows)
    silhouette = silhouette_score(metric_rows, model.predict(metric_rows))
    elbow_results.append({"k": k, "inertia": model.inertia_, "silhouette": silhouette})
    print(f"k={k:2d}   inertia={model.inertia_:>12,.0f}   silhouette={silhouette:.3f}")
elbow = pd.DataFrame(elbow_results)

# Elbow rule of thumb: how much did inertia fall compared with one fewer cluster?
# The elbow is the last k BEFORE the fall drops under 10%.
elbow["inertia_drop_pct"] = -elbow["inertia"].pct_change() * 100
elbow_k = int(elbow.loc[elbow["inertia_drop_pct"] < 10, "k"].iloc[0] - 1)

# Silhouette rule: best silhouette among k >= K_MINIMUM
candidates   = elbow[elbow["k"] >= K_MINIMUM]
silhouette_k = int(candidates.loc[candidates["silhouette"].idxmax(), "k"])

OPTIMAL_K = silhouette_k
print(f"\nElbow suggests k = {elbow_k}.  Best silhouette for k >= {K_MINIMUM} is k = {silhouette_k}.  ->  OPTIMAL_K = {OPTIMAL_K}")
print("(If the two disagree, prefer the one the business can act on; both are reasonable.)")

fig = go.Figure()
fig.add_trace(go.Scatter(x=elbow["k"], y=elbow["inertia"],    name="Inertia (lower = tighter)",       mode="lines+markers"))
fig.add_trace(go.Scatter(x=elbow["k"], y=elbow["silhouette"], name="Silhouette (higher = more distinct)", mode="lines+markers", yaxis="y2"))
fig.add_vline(x=OPTIMAL_K, line_dash="dash", annotation_text=f"chosen k={OPTIMAL_K}")
fig.update_layout(title="Choosing the number of clusters", xaxis_title="number of clusters (k)",
                  yaxis=dict(title="Inertia"), yaxis2=dict(title="Silhouette", overlaying="y", side="right"), height=400)
fig.show()

### Fit the final K-Means on the full training set and read the clusters in plain units

In [ ]:
kmeans = KMeans(n_clusters=OPTIMAL_K, n_init=KMEANS_N_INIT, random_state=RANDOM_SEED).fit(X_train)
raw_train["kmeans_cluster"] = kmeans.labels_

print("Share of training customers in each cluster:")
print(raw_train["kmeans_cluster"].value_counts(normalize=True).sort_index().mul(100).round(1).astype(str).add(" %").to_string())

# kmeans.cluster_centers_ are in z-score units. Undo the scaling (z * std + mean) so a human can read them.
centroid_profile = pd.DataFrame(kmeans.cluster_centers_ * scaler_std.values + scaler_mean.values, columns=INPUT_FEATURES)
centroid_profile.index.name = "cluster"
print("\nTypical customer in each cluster (centroid, original units):")
centroid_profile.round(2)

### Are the clusters really different, and what do we call them?

A cluster is only useful if it is clearly *unlike* the others. Two checks:

* **Heatmap of z-scores** — each cell says how far that cluster's centre sits from the overall average,
  in standard deviations. Deep red = far above average, deep blue = far below. The strongly coloured
  cells are the cluster's "personality" and are what we name it after.
* **Distance between centroids** — in z-score units. Two centroids less than ~1 apart are probably
  one persona split in two; consider lowering `k`.

The automatic names below use the two most extreme features. Treat them as a draft and rename by hand
once you have looked at the heatmap.

In [ ]:
z_profile = pd.DataFrame(kmeans.cluster_centers_, columns=INPUT_FEATURES).round(2)

fig = px.imshow(z_profile, text_auto=True, color_continuous_scale="RdBu_r", zmin=-2, zmax=2, aspect="auto",
                title="What makes each cluster special? (z-scores: red = above average, blue = below)")
fig.update_layout(height=120 + 60 * OPTIMAL_K, xaxis_title="feature", yaxis_title="cluster")
fig.show()

print("Distance between every pair of centroids (z-score units):")
from itertools import combinations
for a, b in combinations(range(OPTIMAL_K), 2):
    distance = np.linalg.norm(kmeans.cluster_centers_[a] - kmeans.cluster_centers_[b])
    print(f"  cluster {a} <-> cluster {b}: {distance:.2f}" + ("   <-- very close, maybe the same persona" if distance < 1 else ""))

def draft_name(z_row):
    # Pick the two features where this cluster deviates most from average, say whether high or low
    top_two = z_row.abs().sort_values(ascending=False).index[:2]
    return " & ".join(f"{'high' if z_row[f] > 0 else 'low'} {f.replace('_', ' ')}" for f in top_two)

KMEANS_SEGMENT_NAMES = {int(c): draft_name(z_profile.loc[c]) for c in z_profile.index}
print("\nDraft cluster names (edit this dict by hand if you prefer business language):")
for cluster, name in KMEANS_SEGMENT_NAMES.items():
    print(f"  cluster {cluster}: {name}")

## 6. Score K-Means on the validation set

`kmeans.predict` assigns each validation customer to the **nearest centroid** learned on train.
The centroids are not re-fitted. If the share of customers per cluster matches train, the
clustering is stable — it found a structure that exists in the population, not just in one sample.

In [ ]:
raw_valid["kmeans_cluster"] = kmeans.predict(X_valid)

cluster_shares = pd.DataFrame({
    "train_%": raw_train["kmeans_cluster"].value_counts(normalize=True).mul(100),
    "valid_%": raw_valid["kmeans_cluster"].value_counts(normalize=True).mul(100),
}).round(1).sort_index()
print("Cluster share, train vs validation (similar numbers = stable clustering):")
print(cluster_shares.to_string())

## 7. Evaluate K-Means — statistics *and* business sense

**Statistical metrics** only look at the feature space (no outcome needed):

| metric | what it measures | good direction |
|---|---|---|
| Silhouette | am I closer to my own cluster than the next one? | higher (max 1) |
| Davies-Bouldin | average "cluster spread ÷ distance to nearest cluster" | lower |
| Calinski-Harabasz | between-cluster spread ÷ within-cluster spread | higher |

**Business metrics** use the performance variables. The question is simple: *do customers in different
clusters actually pay back at different rates?* If every cluster recovers ~77% like the base rate,
the clustering is statistically pretty but useless for collections.

* **Recovery rate per cluster** and **lift vs base rate** (cluster rate ÷ overall rate; 1.5 = this cluster pays 50% more often than average).
* **Recovery separation** — one number for the whole segmentation: the size-weighted spread of recovery
  rates across segments. 0 means every segment has the same rate; bigger means the segments tell them apart better.

**Random baseline.** We also assign customers to segments *at random* (same number of segments) and
compute the same metrics. Any real segmentation must beat this clearly, otherwise the lift we see is noise.

In [ ]:
def statistical_quality(X, labels, label):
    # Compute the three textbook metrics on a 20k subsample (silhouette compares every pair of rows)
    idx = np.random.choice(len(X), min(METRIC_ROWS, len(X)), replace=False)
    result = {"silhouette":        silhouette_score(X[idx], labels[idx]),
              "davies_bouldin":    davies_bouldin_score(X[idx], labels[idx]),
              "calinski_harabasz": calinski_harabasz_score(X[idx], labels[idx])}
    print(f"  {label:<14} silhouette={result['silhouette']:.3f}   davies_bouldin={result['davies_bouldin']:.3f}   calinski_harabasz={result['calinski_harabasz']:,.0f}")
    return result

def business_view(df, segment_col):
    # One row per segment: how many customers, how often they paid, how much they owe
    base_rate = df[TARGET].mean()
    view = df.groupby(segment_col).agg(customers=(ID_COL, "size"),
                                        recovery_rate=(TARGET, "mean"),
                                        avg_recovered_pct=("recovery_rate_90d", "mean"),
                                        avg_balance=("outstanding_balance", "mean"))
    view["share_%"]      = view["customers"] / len(df) * 100
    view["lift_vs_base"] = view["recovery_rate"] / base_rate
    return view.round(3)

def recovery_separation(df, segment_col):
    # Size-weighted standard deviation of recovery rate across segments. 0 = segments are indistinguishable.
    per_segment = df.groupby(segment_col)[TARGET].agg(["mean", "size"])
    overall = df[TARGET].mean()
    return float(np.sqrt(np.average((per_segment["mean"] - overall) ** 2, weights=per_segment["size"])))

# ---- Random baseline: same number of segments, assigned by coin flip ------------------------
raw_valid["random_segment"] = np.random.randint(0, OPTIMAL_K, len(raw_valid))

print("Statistical quality on validation (20k-row sample):")
kmeans_stats = statistical_quality(X_valid, raw_valid["kmeans_cluster"].to_numpy(), "K-Means")
random_stats = statistical_quality(X_valid, raw_valid["random_segment"].to_numpy(), "random")

kmeans_business = business_view(raw_valid, "kmeans_cluster")
kmeans_business["name"] = kmeans_business.index.map(KMEANS_SEGMENT_NAMES)
print("\nBusiness view of K-Means clusters on validation:")
display(kmeans_business)

kmeans_sep = recovery_separation(raw_valid, "kmeans_cluster")
random_sep = recovery_separation(raw_valid, "random_segment")
print(f"Recovery separation:  K-Means = {kmeans_sep:.4f}   random = {random_sep:.4f}   "
      f"->  K-Means is {kmeans_sep / max(random_sep, 1e-9):.0f}x better than chance")

fig = px.bar(kmeans_business.reset_index(), x="kmeans_cluster", y="recovery_rate", text="name", color="share_%",
             title="Recovery rate by K-Means cluster (validation) — do the personas pay back differently?")
fig.add_hline(y=raw_valid[TARGET].mean(), line_dash="dash", annotation_text="overall rate")
fig.update_layout(height=420, xaxis_title="cluster", yaxis_title="share recovered in 90 days")
fig.show()

### Picture of the clusters

We have 9 features, so we cannot plot customers directly. **PCA** squashes the 9 dimensions down to the
2 directions along which customers vary the most, so we can draw a scatter. The printed percentage says
how much of the total variation those 2 directions keep; if it is low (< 50%) the picture is only a rough sketch.

In [ ]:
plot_idx = np.random.choice(len(X_valid), 20_000, replace=False)          # 20k dots is plenty for a scatter
pca = PCA(n_components=2, random_state=RANDOM_SEED).fit(X_train[:100_000])
coords = pca.transform(X_valid[plot_idx])

plot_df = pd.DataFrame(coords, columns=["pc1", "pc2"])
plot_df["cluster"] = raw_valid["kmeans_cluster"].to_numpy()[plot_idx].astype(str)
print(f"The 2 PCA directions keep {pca.explained_variance_ratio_.sum():.0%} of the variation in the 9 features")

fig = px.scatter(plot_df, x="pc1", y="pc2", color="cluster", opacity=0.4,
                 title="K-Means clusters on validation customers (PCA projection)")
fig.update_traces(marker_size=4)
fig.update_layout(height=500)
fig.show()

## 8. Shallow decision tree — segments built from the outcome

Now the opposite approach. We *tell* the model who paid (`recovered_90d`) and let it find the
yes/no questions on the input features that best separate payers from non-payers. Depth 3 means at
most three questions, so every customer lands in one of at most 8 **leaves** — each leaf is a segment
with a rule a collections manager can read aloud.

Two guard rails: `TREE_MIN_LEAF_FRAC` stops the tree carving out tiny segments, and we check the AUC
(how well the tree ranks payers above non-payers) on train vs validation — if validation is much worse
the tree memorised the training set.

In [ ]:
min_leaf_rows = int(TREE_MIN_LEAF_FRAC * len(raw_train))
tree = DecisionTreeClassifier(max_depth=TREE_MAX_DEPTH, min_samples_leaf=min_leaf_rows, random_state=RANDOM_SEED)
tree.fit(raw_train[INPUT_FEATURES], raw_train[TARGET])          # trees don't need scaled features; raw units give readable rules

# tree.apply returns the leaf number each customer ends up in -> that leaf number IS the segment
raw_train["tree_segment"] = tree.apply(raw_train[INPUT_FEATURES])
raw_valid["tree_segment"] = tree.apply(raw_valid[INPUT_FEATURES])
print(f"Tree has {tree.get_n_leaves()} leaves = {tree.get_n_leaves()} segments\n")
print(export_text(tree, feature_names=INPUT_FEATURES, decimals=1))

train_auc = roc_auc_score(raw_train[TARGET], tree.predict_proba(raw_train[INPUT_FEATURES])[:, 1])
valid_auc = roc_auc_score(raw_valid[TARGET], tree.predict_proba(raw_valid[INPUT_FEATURES])[:, 1])
print(f"AUC   train = {train_auc:.3f}   valid = {valid_auc:.3f}   (0.5 = coin flip, 1.0 = perfect; close together = not overfitting)")

### Name each leaf by its rule, then evaluate it the same way we evaluated K-Means

In [ ]:
def leaf_rules(tree, feature_names):
    # Walk from the root to every leaf, remembering the tightest lower/upper bound seen for each feature,
    # so a leaf reads "43 < days_past_due <= 70" instead of "days_past_due <= 70 & days_past_due > 43".
    t = tree.tree_
    rules = {}
    def walk(node, bounds):
        if t.children_left[node] == -1:                     # no children -> this is a leaf
            parts = []
            for f, (lo, hi) in bounds.items():
                if lo is not None and hi is not None: parts.append(f"{lo:.1f} < {f} <= {hi:.1f}")
                elif hi is not None:                  parts.append(f"{f} <= {hi:.1f}")
                else:                                 parts.append(f"{f} > {lo:.1f}")
            rules[node] = " & ".join(parts) or "all customers"
            return
        f, threshold = feature_names[t.feature[node]], t.threshold[node]
        left  = {k: list(v) for k, v in bounds.items()}     # copy so siblings don't share bounds
        right = {k: list(v) for k, v in bounds.items()}
        left.setdefault(f, [None, None])[1]  = threshold     # going left means  f <= threshold
        right.setdefault(f, [None, None])[0] = threshold     # going right means f >  threshold
        walk(t.children_left[node], left)
        walk(t.children_right[node], right)
    walk(0, {})
    return rules

TREE_SEGMENT_NAMES = leaf_rules(tree, INPUT_FEATURES)

tree_business = business_view(raw_valid, "tree_segment")
tree_business["rule"] = tree_business.index.map(TREE_SEGMENT_NAMES)
print("Business view of tree segments on validation:")
display(tree_business)

tree_sep = recovery_separation(raw_valid, "tree_segment")
raw_valid["random_segment_tree"] = np.random.randint(0, tree.get_n_leaves(), len(raw_valid))   # random floor with the tree's segment count
random_sep_tree = recovery_separation(raw_valid, "random_segment_tree")
print(f"Recovery separation:  tree = {tree_sep:.4f}   random = {random_sep_tree:.4f}   ->  tree is {tree_sep / max(random_sep_tree, 1e-9):.0f}x better than chance")

# Apples-to-apples with K-Means: treat the leaves as clusters in the scaled feature space.
# Expect these to be WORSE than K-Means -- the tree never tried to make customers in a leaf look alike.
print("\nStatistical quality of tree leaves treated as clusters:")
tree_stats = statistical_quality(X_valid, raw_valid["tree_segment"].to_numpy(), "tree")

fig = px.bar(tree_business.reset_index().sort_values("recovery_rate"), x="rule", y="recovery_rate", color="share_%",
             title="Recovery rate by decision-tree segment (validation)")
fig.add_hline(y=raw_valid[TARGET].mean(), line_dash="dash", annotation_text="overall rate")
fig.update_layout(height=450, xaxis_tickangle=-20, yaxis_title="share recovered in 90 days")
fig.show()

## 9. Compare the two methods (and random)

One table, three columns. Read it row by row:

* **silhouette / davies_bouldin / calinski_harabasz** — K-Means should win; it was built to make
  customers within a cluster look alike. The tree never tried.
* **recovery_separation / max_lift / min_lift** — the tree usually wins; it was built from the outcome.
  The interesting question is *by how much*, and whether K-Means is also well clear of random.
* **smallest_segment_%** — a segment under ~5% is hard to build a strategy around.

The cross-tab below shows how the two segmentations overlap: if a K-Means persona spreads across
several tree leaves, that persona contains both payers and non-payers, i.e. it describes *who the
customer is* but not *whether they will pay*.

In [ ]:
def summary(stats, sep, n_segments, business):
    return {**stats, "n_segments": n_segments, "recovery_separation": sep,
            "smallest_segment_%": business["share_%"].min(),
            "max_lift": business["lift_vs_base"].max(), "min_lift": business["lift_vs_base"].min()}

random_business = business_view(raw_valid, "random_segment")
comparison = pd.DataFrame({
    "K-Means":       summary(kmeans_stats, kmeans_sep, OPTIMAL_K, kmeans_business),
    "Decision tree": summary(tree_stats, tree_sep, tree.get_n_leaves(), tree_business),
    "Random":        summary(random_stats, random_sep, OPTIMAL_K, random_business),
}).round(3)
print("Validation comparison  (silhouette, calinski, separation, max_lift: higher is better;  davies_bouldin, min_lift: lower is better)")
display(comparison)

print("\n% of each K-Means cluster that falls in each tree leaf (rows sum to 100):")
overlap = pd.crosstab(raw_valid["kmeans_cluster"], raw_valid["tree_segment"], normalize="index").mul(100).round(0)
display(overlap)

winner = "Decision tree" if tree_sep > kmeans_sep else "K-Means"
print(f"\n=> Strongest recovery separation: {winner}  ({max(tree_sep, kmeans_sep):.4f} vs {min(tree_sep, kmeans_sep):.4f};  random = {random_sep:.4f})")

### Recommendation

Both methods are "right" at what they were built for, so the choice depends on what the segments are used for:

* **Use the decision tree** when the segments drive *prioritisation* — who to call first, where to spend
  agent time. Its segments are built from recovery, come as plain rules anyone can audit, and are easy to
  re-implement in a rules engine. Condition: its recovery separation beats K-Means by a clear margin
  *and* no leaf is below ~5%.
* **Use K-Means** when the segments drive *treatment style* — tone, channel, settlement offers. Its
  personas describe the whole customer (balance, tenure, engagement), are stable between train and
  validation, don't depend on one target definition, and still rank clearly on recovery (see lift).
  Condition: its recovery separation is within ~10% of the tree's and far above random.
* **Many teams use both**: K-Means persona for *how* to approach, tree leaf for *how urgently*. The
  overlap table above is the map between them.

The cell below records the choice (`FINAL_METHOD` — override by hand if the business view differs
from the automatic winner) and saves every artifact the scoring job needs.

In [ ]:
FINAL_METHOD = winner                      # "K-Means" or "Decision tree" -- override here if you disagree
FINAL_NAMES  = KMEANS_SEGMENT_NAMES if FINAL_METHOD == "K-Means" else TREE_SEGMENT_NAMES
print(f"FINAL_METHOD = {FINAL_METHOD}  ({len(FINAL_NAMES)} segments)")
for segment, name in FINAL_NAMES.items():
    print(f"  {segment}: {name}")

# Centroids are saved in z-score units; the scoring job scales new customers the same way before measuring distance
json.dump({"k": OPTIMAL_K,
           "centroids_scaled": kmeans.cluster_centers_.tolist(),
           "names": {str(c): n for c, n in KMEANS_SEGMENT_NAMES.items()}},
          open(PATH_CENTROIDS, "w"), indent=2)
pickle.dump(tree, open(PATH_TREE, "wb"))
json.dump({"method": FINAL_METHOD, "names": {str(s): n for s, n in FINAL_NAMES.items()}}, open(PATH_FINAL_CHOICE, "w"), indent=2)
print(f"\nSaved: {PATH_CENTROIDS}, {PATH_TREE}, {PATH_FINAL_CHOICE}")

## 10. Standalone scoring pipeline

This cell depends on **nothing above except the files on disk** — copy it into a scheduled job.
It replays the same three preparation steps with the *saved* numbers (never re-computed), then
assigns each customer to the nearest saved centroid.

The last lines are a sanity check: scoring the validation customers from disk must give exactly the
labels we computed in memory. 100% agreement means the artifacts are complete and the pipeline is faithful.

In [ ]:
import json
import numpy as np
import pandas as pd

def score_kmeans_segments(new_customers: pd.DataFrame,
                          scaler_path="artifacts/scaler.json",
                          centroid_path="artifacts/centroids.json") -> pd.DataFrame:
    """Assign each customer to a saved K-Means segment using only the artifacts on disk."""
    scaler    = json.load(open(scaler_path))
    saved     = json.load(open(centroid_path))
    centroids = np.array(saved["centroids_scaled"])          # shape (k, n_features), z-score units
    names     = saved["names"]
    features  = scaler["features"]
    print(f"Scoring {len(new_customers):,} customers against {len(centroids)} centroids on {len(features)} features")

    X = new_customers[features].copy()
    X = X.fillna(pd.Series(scaler["median"]))                                      # step 1: same imputation as training
    X = X.clip(pd.Series(scaler["lower"]), pd.Series(scaler["upper"]), axis=1)     # step 2: same outlier clipping
    X = (X - pd.Series(scaler["mean"])) / pd.Series(scaler["std"])                 # step 3: same z-scaling

    # Distance from every customer to every centroid -> matrix of shape (n_customers, k); nearest wins
    distances = np.linalg.norm(X.to_numpy()[:, None, :] - centroids[None, :, :], axis=2)

    result = pd.DataFrame(index=new_customers.index)
    if "customer_id" in new_customers:
        result["customer_id"] = new_customers["customer_id"]
    result["segment"]              = distances.argmin(axis=1)
    result["segment_name"]         = result["segment"].astype(str).map(names)
    result["distance_to_centroid"] = distances.min(axis=1).round(3)     # large values = unusual customer, worth a look

    print("Segment distribution:")
    print(result["segment_name"].value_counts(normalize=True).mul(100).round(1).astype(str).add(" %").to_string())
    return result

# ---- Sanity check against the in-memory labels ----------------------------------------------
check_ids = raw_valid[ID_COL].head(50_000)
check_rows = pd.read_csv(PATH_INPUT).set_index(ID_COL).loc[check_ids].reset_index()   # reload from disk, untouched
scored = score_kmeans_segments(check_rows)
agreement = (scored["segment"].to_numpy() == raw_valid["kmeans_cluster"].head(50_000).to_numpy()).mean()
print(f"\nAgreement with in-notebook K-Means labels: {agreement:.1%}  (should be 100%)")
scored.head()